# Demo 05 — DataTransformer and Entity Embeddings

This notebook demonstrates how the **DataTransformer** encodes tabular data for GAN training. It shows:

1. How low-cardinality categoricals are one-hot encoded (OHE)
2. How high-cardinality categoricals switch to **entity embeddings**
3. The fit / transform / inverse-transform round-trip

No external data files or Spark are required.

In [ ]:
import numpy as np
import pandas as pd

from syntho_hive.core.data.transformer import DataTransformer
from syntho_hive.interface.config import Metadata

## 1. Create Sample Data

We generate a 120-row product table. The `brand` column has 25 unique values (above the embedding threshold of 20), so it will use entity embeddings instead of OHE.

In [ ]:
def make_data(num_rows: int = 120, high_card: int = 25) -> pd.DataFrame:
    rng = np.random.default_rng(100)
    df = pd.DataFrame(
        {
            "product_id": np.arange(1, num_rows + 1),
            "category": rng.choice(
                ["electronics", "apparel", "home", "toys"],
                size=num_rows,
                p=[0.35, 0.25, 0.25, 0.15],
            ),
            "brand": rng.choice(
                [f"brand_{i}" for i in range(high_card)], size=num_rows
            ),
            "price": rng.normal(80, 25, size=num_rows).clip(5, 300).round(2),
            "inventory": rng.integers(0, 500, size=num_rows),
        }
    )
    return df

EMBEDDING_THRESHOLD = 20
df = make_data(num_rows=120, high_card=EMBEDDING_THRESHOLD + 5)
print(f"Data shape: {df.shape}")
print(f"Unique brands: {df['brand'].nunique()}")
print(f"Unique categories: {df['category'].nunique()}")
df.head()

## 2. Define Metadata

We mark `product_id` as the primary key (excluded from transformation) and `brand` as a high-cardinality column.

In [ ]:
meta = Metadata()
meta.add_table(
    name="products",
    pk="product_id",
    high_cardinality_cols=["brand"],
)

## 3. Fit and Transform

The `DataTransformer` drops PK/FK columns, applies Variational Gaussian Mixture (VGM) normalization to continuous columns, OHE to low-cardinality categoricals, and entity embeddings to high-cardinality categoricals. The result is a dense NumPy matrix.

In [ ]:
transformer = DataTransformer(
    metadata=meta, embedding_threshold=EMBEDDING_THRESHOLD
)

print("Fitting transformer (drops PK/FK automatically)...")
transformer.fit(df, table_name="products")

transformed = transformer.transform(df)
print(f"Original shape:    {df.shape}")
print(f"Transformed shape: {transformed.shape}")
print(f"\nFirst 3 rows of the dense matrix:")
print(transformed[:3])

## 4. Inverse Transform

The `inverse_transform()` method converts the dense matrix back to a human-readable DataFrame. Note that the PK column is not included in the round-trip — we re-attach it manually.

In [ ]:
recovered = transformer.inverse_transform(transformed)
recovered.insert(0, "product_id", range(1, len(recovered) + 1))

print(f"Recovered shape: {recovered.shape}")
recovered.head()

## 5. Round-Trip Comparison

Compare the original and recovered data to see the fidelity of the transform/inverse-transform cycle.

In [ ]:
print("=== Original ===")
print(df.head())
print("\n=== Recovered ===")
print(recovered.head())